# Synthetic Data Generator for Bank Queue Management

This notebook generates synthetic data simulating customer arrivals and transactions in a bank branch queue system.

The data includes customer details, transaction types, and queue timing to support queue management analysis.

**Key Features:**
- Customizable dataset size (default 10000 rows)
- Locale set to English (India) (`en_IN`)
- Tunable parameters for demographics and transaction types
- Output file: `syntheticdata.csv`

---

### How to use this notebook:
1. Modify parameters in the `generate_synthetic_data` function call below to experiment with different synthetic data profiles.
2. Run all cells to generate the dataset.
3. Download `syntheticdata.csv` or convert for further analysis.

This notebook can be run locally or on Google Colab.

In [ ]:
# Install dependencies (if running in Colab)
!pip install faker pandas --quiet

In [ ]:
import csv
import random
from datetime import datetime, timedelta
from faker import Faker
import pandas as pd


In [ ]:
def generate_synthetic_data(
        num_records=10000,
        output_file='syntheticdata.csv',
        locale='en_IN',
        gender_dist=None,
        transaction_types=None,
        work_start_hour=9,
        work_end_hour=17,
        avg_service_time_sec=300,
        service_time_std_dev=120
    ):
    """
    Generate synthetic bank queue dataset.

    Parameters:
        num_records (int): Number of rows to generate.
        output_file (str): CSV output filename.
        locale (str): Faker locale.
        gender_dist (dict): Distribution of genders.
        transaction_types (dict): Transaction types and probabilities.
        work_start_hour, work_end_hour (int): Bank working hours.
        avg_service_time_sec, service_time_std_dev (int): Service time parameters.
    """
    try:
        fake = Faker(locale)
        Faker.seed(42)
        random.seed(42)

        if gender_dist is None:
            gender_dist = {'Male': 0.49, 'Female': 0.49, 'Other': 0.02}
        if transaction_types is None:
            transaction_types = {
                'Cash Deposit': 0.3,
                'Cash Withdrawal': 0.3,
                'Loan Inquiry': 0.1,
                'Account Opening': 0.1,
                'Cheque Deposit': 0.1,
                'Others': 0.1
            }

        gender_choices = list(gender_dist.keys())
        gender_probs = list(gender_dist.values())
        transaction_type_choices = list(transaction_types.keys())
        transaction_type_probs = list(transaction_types.values())

        header = ['customer_id', 'name', 'age', 'gender', 'city', 'transaction_type',
                  'transaction_amount', 'arrival_time', 'service_start_time', 'service_duration_sec']

        base_date = datetime.now().date()
        data = []
        last_service_end = None

        for i in range(num_records):
            customer_id = i + 1
            name = fake.name()
            age = random.randint(18, 85)
            gender = random.choices(gender_choices, gender_probs)[0]
            city = fake.city()

            transaction_type = random.choices(transaction_type_choices, transaction_type_probs)[0]
            if transaction_type in ['Cash Deposit', 'Cash Withdrawal', 'Cheque Deposit']:
                transaction_amount = round(random.uniform(100, 50000), 2)
            elif transaction_type == 'Loan Inquiry':
                transaction_amount = 0.0
            elif transaction_type == 'Account Opening':
                transaction_amount = round(random.uniform(1000, 10000), 2)
            else:
                transaction_amount = round(random.uniform(100, 20000), 2)

            arrival_hour = random.randint(work_start_hour, work_end_hour - 1)
            arrival_minute = random.randint(0, 59)
            arrival_second = random.randint(0, 59)
            arrival_time = datetime.combine(
                base_date, datetime.min.time()) + timedelta(hours=arrival_hour, minutes=arrival_minute, seconds=arrival_second)

            if last_service_end is None or arrival_time > last_service_end:
                service_start_time = arrival_time
            else:
                service_start_time = last_service_end

            service_duration_sec = max(60, int(random.gauss(avg_service_time_sec, service_time_std_dev)))
            last_service_end = service_start_time + timedelta(seconds=service_duration_sec)

            data.append([
                customer_id, name, age, gender, city, transaction_type, transaction_amount,
                arrival_time.strftime('%Y-%m-%d %H:%M:%S'),
                service_start_time.strftime('%Y-%m-%d %H:%M:%S'),
                service_duration_sec
            ])

        df = pd.DataFrame(data, columns=header)
        df.to_csv(output_file, index=False)
        print(f"Generated {num_records} records and saved to '{output_file}'")
        return df
    except Exception as e:
        print(f"Error generating synthetic data: {e}")



### Generate 10,000 synthetic records (default)

You can modify `num_records` to generate a smaller or larger dataset.


In [ ]:
df_synthetic = generate_synthetic_data(num_records=10000)

### Preview of the generated data

Check the first few rows to understand the data structure.

In [ ]:
df_synthetic.head()

### Instructions for students
- You may change parameters inside `generate_synthetic_data` such as gender distribution, transaction type probabilities, and working hours.
- Rerun the data generation cell to produce different queue scenarios.
- Use this data for queue analysis, ML modeling, or simulation.